# curious-george — Phase 0 validation notebook

Runs the already-validated foundation (memory store, fabricated ground truth, loss measurement, LoRA fine-tuning) against GPU compute, then moves on to Phase 1 experiments. See the repo [README](https://github.com/stvenmobile/curious-george#readme) for the full roadmap and the definitions behind each term used below.

In [ ]:
!git clone https://github.com/stvenmobile/curious-george.git
%cd curious-george
!pip install -q -r requirements.txt

In [ ]:
import sys
sys.path.insert(0, "src")

from curious_george.warble_harness import run_sanity_check
run_sanity_check(device="cuda")

## Persisting memory across sessions

Colab's local disk is wiped on every runtime restart. `MemoryStore`'s whole reason for existing is a `loss_history` that survives across sessions — without that, `learning_progress()` never has more than one measurement to compare against, and the "learn and retain" premise of the project doesn't hold.

This cell mounts Google Drive and points `MemoryStore.save()`/`load()` at a folder there via the `CURIOUS_GEORGE_MEMORY_DIR` environment variable, instead of the ephemeral local `data/memory/` default. Run it once per session, before any `save()`/`load()` call with no explicit path — it's read at call time, so it doesn't need to happen in the same cell as those calls.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ["CURIOUS_GEORGE_MEMORY_DIR"] = "/content/drive/MyDrive/curious-george-memory"

from curious_george.memory_store import MemoryStore
store = MemoryStore.load()   # picks up last session's state from Drive, or starts empty on first run
print(f"Loaded {len(store)} item(s) from persistent memory.")

## The genuine learning test: LoRA fine-tuning, cold measurement

The sanity check above proved the plumbing works, but its "study" step just placed warble facts directly in the prompt — a confound, since *any* coherent context measurably helps next-token prediction (which is exactly why quaddles, whose own facts were never shown, still improved: +1.15 nats of that improvement is just reading-comprehension priming, not learning).

`run_lora_check` fixes this. It fine-tunes a small LoRA adapter (the base model's own weights are frozen and never change) on warble facts only, then measures loss on **cold** prompts — no warble content anywhere in the prompt, before or after fine-tuning. Any improvement can only come from the adapter's trained weights, not from priming. The original CPU run (`piper_assistant`, pre-port) took ~41 minutes and found: warbles improved 2.65 nats cold, quaddles (never trained on, structurally similar) leaked 0.68 nats (a ~26% residual confound from shared sentence structure between the two fabricated species — see the README's Phase 4). This run should reproduce that on GPU, much faster.

In [ ]:
from curious_george.warble_harness import run_lora_check
run_lora_check(device="cuda")

### Result (2026-09-12, T4 GPU)

| | before | after | delta | accuracy |
|---|---|---|---|---|
| warbles (LoRA-trained) | 3.4684 | 0.7177 | **+2.7507** | 0.483 -> 0.913 |
| quaddles (never trained on) | 3.6189 | 3.0267 | **+0.5922** | 0.271 -> 0.435 |

Leak ratio 0.215 — reproduces the original CPU run's finding (2.65 / 0.68 nats, ratio 0.258) almost exactly, on different hardware, with accuracy landing on the identical 0.913 / 0.435 both times. Confirms the port is sound and the result isn't hardware-specific.

Same interpretation as before: genuine, persistent, cold, mostly fact-specific learning from the LoRA adapter's own weights — real evidence for the core mechanism — with a real, if smaller, residual leak from warbles and quaddles sharing sentence structure. That leak is the open question Phase 4 in the README exists to address; Phase 1 (defining and testing a curiosity score) is the more immediate next step.